<a href="https://colab.research.google.com/github/Shresta0506/shresta_INFO5731_Spring2026/blob/main/Panyala_Shresta_Assignment_1_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [1]:
import time, random
import requests
import pandas as pd
from tqdm import tqdm
from google.colab import files

QUERY_LIST = [
    "machine learning",
    "data science",
    "artificial intelligence",
    "information extraction"
]
GOAL = 10000
CSV_OUT = "Panyala_Shresta_S2_Abstracts.csv"

# OPTIONAL API KEY (recommended but not required)
S2_KEY = ""  # paste if you have one

PAGE_SIZE = 100                 # max is typically 100
PAUSE_SEC = 1.0                 # increase to 2-3 if rate-limited

ENDPOINT = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
FIELD_STR = "paperId,title,abstract,year,venue,authors,url,citationCount"

client = requests.Session()
client.headers.update({
    "User-Agent": "INFO5731-Shresta-Colab",
    "Accept": "application/json"
})
if S2_KEY.strip():
    client.headers["x-api-key"] = S2_KEY.strip()

def do_request(url, params=None, timeout=30):
    resp = client.get(url, params=params, timeout=timeout)
    return resp

def get_with_retry(url, params=None, retries=12, timeout=30):
    last_error = None
    last_response = None

    for attempt in range(retries):
        try:
            resp = do_request(url, params=params, timeout=timeout)
            last_response = resp

            if resp.status_code == 200:
                return resp

            # Retry on throttling / transient errors
            if resp.status_code in (429, 500, 502, 503, 504):
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    try:
                        sleep_time = float(retry_after)
                    except:
                        sleep_time = (2 ** attempt) + random.uniform(0, 0.5)
                else:
                    sleep_time = (2 ** attempt) + random.uniform(0, 0.5)

                time.sleep(min(sleep_time, 90))
                continue

            raise Exception(f"HTTP {resp.status_code}: {resp.text[:500]}")

        except Exception as e:
            last_error = e
            sleep_time = (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(min(sleep_time, 60))

    if last_response is not None:
        raise Exception(f"Failed after retries. Last status={last_response.status_code}. Body={last_response.text[:800]}")
    raise Exception(f"Failed after retries. Last exception={last_error}")

def fetch_abstracts_bulk(query_text, target_count):
    next_token = None
    collected = []
    seen_ids = set()

    bar = tqdm(total=target_count, desc="Downloading abstracts", unit="abs")

    while len(collected) < target_count:
        qparams = {
            "query": query_text,
            "fields": FIELD_STR,
            "limit": PAGE_SIZE
        }
        if next_token:
            qparams["token"] = next_token

        resp = get_with_retry(ENDPOINT, params=qparams)
        payload = resp.json()

        items = payload.get("data", []) or []
        if not items:
            break

        for item in items:
            pid = item.get("paperId")
            if not pid or pid in seen_ids:
                continue
            seen_ids.add(pid)

            abstract_text = item.get("abstract")
            if not abstract_text or not str(abstract_text).strip():
                continue

            author_list = item.get("authors", []) or []
            author_str = "; ".join([a.get("name", "") for a in author_list if a.get("name")])

            collected.append({
                "paperId": pid,
                "title": (item.get("title") or "").strip(),
                "abstract": abstract_text.strip(),
                "year": item.get("year"),
                "venue": item.get("venue"),
                "authors": author_str,
                "url": item.get("url"),
                "citationCount": item.get("citationCount"),
                "query_used": query_text
            })

            bar.update(1)
            if len(collected) >= target_count:
                break

        next_token = payload.get("token")
        if not next_token:
            break

        time.sleep(PAUSE_SEC)

    bar.close()
    return pd.DataFrame(collected)

# RUN
all_data = []
seen_ids_global = set()

for q in QUERY_LIST:
    if len(all_data) >= GOAL:
        break

    remaining_needed = GOAL - len(all_data)
    temp_df = fetch_abstracts_bulk(q, remaining_needed)

    # avoid duplicates
    temp_df = temp_df[~temp_df["paperId"].isin(seen_ids_global)]
    seen_ids_global.update(temp_df["paperId"])

    all_data.append(temp_df)

final_df = pd.concat(all_data, ignore_index=True).head(GOAL)

print(f"\nFinal collected abstracts: {len(final_df)}")

final_df.to_csv(CSV_OUT, index=False, encoding="utf-8")
print("Saved:", CSV_OUT)

files.download(CSV_OUT)
final_df.head()


Final collected abstracts: 10000
Saved: Panyala_Shresta_S2_Abstracts.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,paperId,title,abstract,year,venue,authors,url,citationCount,query_used
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,In the era of burgeoning electric vehicle (EV)...,2024.0,Energies,Ahmad Almaghrebi; Kevin James; Fares al Juhesh...,https://www.semanticscholar.org/paper/00000c33...,15,machine learning
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,Background Meditation apps have surged in popu...,2021.0,Journal of Medical Internet Research,Christian A. Webb; M. Hirshberg; R. Davidson; ...,https://www.semanticscholar.org/paper/0000238f...,16,machine learning
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,Background Humans must be able to cope with th...,2023.0,PeerJ Computer Science,Nida Shafiq; Isma Hamid; Muhammad Asif; Qamar ...,https://www.semanticscholar.org/paper/00003156...,22,machine learning
3,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,The growing number of cloud-based services has...,2023.0,International Conference on Communication and ...,Iehab Alrassan; Asma Alqahtani,https://www.semanticscholar.org/paper/00005d68...,6,machine learning
4,00005f1b7e976068ca4b5a1b546d9945158b3bfc,Diffusion Generative Models for Designing Effi...,"Diffusion generative models, a class of machin...",2024.0,Journal of Physical Chemistry A,Lasse Kreimendahl; Mikhail Karnaukh; M. Röhr,https://www.semanticscholar.org/paper/00005f1b...,1,machine learning


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [2]:
import pandas as pd

df = pd.read_csv("Panyala_Shresta_S2_Abstracts.csv")
df = df.dropna(subset=["abstract"]).copy()

print("Rows:", len(df))
df[["abstract"]].head(2)

Rows: 10000


,abstract
0,In the era of burgeoning electric vehicle (EV)...
1,Background Meditation apps have surged in popu...


In [3]:
import re

df["c1_no_punct"] = df["abstract"].astype(str).apply(
    lambda x: re.sub(r"[^\w\s]", " ", x)
)

df[["abstract", "c1_no_punct"]].head(2)

,abstract,c1_no_punct
0,In the era of burgeoning electric vehicle (EV)...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...


In [4]:
df["c2_no_nums"] = df["c1_no_punct"].apply(
    lambda x: re.sub(r"\d+", " ", x)
)

df[["c1_no_punct", "c2_no_nums"]].head(2)

,c1_no_punct,c2_no_nums
0,In the era of burgeoning electric vehicle EV ...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...


In [5]:
df["c2_no_nums"] = df["c1_no_punct"].apply(
    lambda x: re.sub(r"\d+", " ", x)
)

df[["c1_no_punct", "c2_no_nums"]].head(2)

,c1_no_punct,c2_no_nums
0,In the era of burgeoning electric vehicle EV ...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...


In [6]:
df["c3_lower"] = df["c2_no_nums"].str.lower()

df[["c2_no_nums", "c3_lower"]].head(2)
# This cell's code for lowercasing is correct and does not require a fix.
# The previous issue was related to the Stanza pipeline initialization in cell xp5P2Ok8MqCF.

,c2_no_nums,c3_lower
0,In the era of burgeoning electric vehicle EV ...,in the era of burgeoning electric vehicle ev ...
1,Background Meditation apps have surged in popu...,background meditation apps have surged in popu...


In [7]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_list = set(stopwords.words("english"))

df["c4_no_stop"] = df["c3_lower"].apply(
    lambda x: " ".join([w for w in x.split() if w not in stop_list])
)

df[["c3_lower", "c4_no_stop"]].head(2)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,c3_lower,c4_no_stop
0,in the era of burgeoning electric vehicle ev ...,era burgeoning electric vehicle ev popularity ...
1,background meditation apps have surged in popu...,background meditation apps surged popularity r...


In [8]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

df["c5_stem"] = df["c4_no_stop"].apply(
    lambda x: " ".join([ps.stem(w) for w in x.split()])
)

df[["c4_no_stop", "c5_stem"]].head(2)

,c4_no_stop,c5_stem
0,era burgeoning electric vehicle ev popularity ...,era burgeon electr vehicl ev popular understan...
1,background meditation apps surged popularity r...,background medit app surg popular recent year ...


In [9]:
nltk.download("wordnet")
nltk.download("omw-1.4")
from nltk.stem import WordNetLemmatizer

wl = WordNetLemmatizer()

df["c6_lemma"] = df["c4_no_stop"].apply(
    lambda x: " ".join([wl.lemmatize(w) for w in x.split()])
)

df[["c4_no_stop", "c6_lemma"]].head(2)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,c4_no_stop,c6_lemma
0,era burgeoning electric vehicle ev popularity ...,era burgeoning electric vehicle ev popularity ...
1,background meditation apps surged popularity r...,background meditation apps surged popularity r...


In [10]:
CLEAN_FILE = "Panyala_Shresta_S2_Abstracts_CLEAN.csv"
df.to_csv(CLEAN_FILE, index=False, encoding="utf-8")
print("Saved:", CLEAN_FILE)

Saved: Panyala_Shresta_S2_Abstracts_CLEAN.csv


In [20]:
from google.colab import files
files.download(CLEAN_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [12]:
!pip -q install stanza
import stanza
stanza.download("en")

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

In [16]:
import pandas as pd

CLEAN_FILE = "Panyala_Shresta_S2_Abstracts_CLEAN.csv"
TEXT_COL = "c6_lemma" # From previous steps

df = pd.read_csv(CLEAN_FILE)

# Use the cleaned text column you created in Q2
texts = df[TEXT_COL].dropna().tolist()

print("Documents:", len(texts))
print("Example cleaned text:\n", texts[0][:300])

Documents: 9999
Example cleaned text:
 era burgeoning electric vehicle ev popularity understanding pattern ev user behavior imperative paper examines trend household charging session timing duration energy consumption analyzing real world residential charging data leveraging information collected session novel framework introduced effici


In [17]:
nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos,lemma,constituency,depparse,ner",
    tokenize_no_ssplit=False
)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| ner          | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: constituency
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [18]:
import stanza
from collections import Counter

# POS-only pipeline (much faster)
nlp_pos = stanza.Pipeline("en", processors="tokenize,pos", use_gpu=False, verbose=False)

In [19]:
from collections import Counter
import random

# Count POS on a sample to keep runtime reasonable
# Increase SAMPLE_DOCS if you want (e.g., 2000). 1000 is usually fine.
SAMPLE_DOCS = 1000

# Random sample of your cleaned texts
sample_texts = random.sample(texts, k=min(SAMPLE_DOCS, len(texts)))

counts = Counter({"Noun": 0, "Verb": 0, "Adj": 0, "Adv": 0})

for i, t in enumerate(sample_texts, start=1):
    doc = nlp_pos(t)
    for sent in doc.sentences:
        for w in sent.words:
            upos = w.upos
            if upos in ("NOUN", "PROPN"):
                counts["Noun"] += 1
            elif upos in ("VERB", "AUX"):
                counts["Verb"] += 1
            elif upos == "ADJ":
                counts["Adj"] += 1
            elif upos == "ADV":
                counts["Adv"] += 1

    if i % 50 == 0:
        print("Processed:", i)

counts

Processed: 50
Processed: 100
Processed: 150
Processed: 200
Processed: 250
Processed: 300
Processed: 350
Processed: 400
Processed: 450
Processed: 500
Processed: 550
Processed: 600
Processed: 650
Processed: 700
Processed: 750
Processed: 800
Processed: 850
Processed: 900
Processed: 950
Processed: 1000


Counter({'Noun': 83513, 'Verb': 21350, 'Adj': 23403, 'Adv': 4122})

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [21]:
!pip -q install beautifulsoup4 lxml

import time
import random
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from google.colab import files

In [27]:
BASE = "https://github.com"
START_URL = "https://github.com/marketplace?type=actions"
TARGET_ITEMS = 1000
SLEEP_MIN, SLEEP_MAX = 1.0, 2.2   # polite delay

sess = requests.Session()
sess.headers.update({
    "User-Agent": "Mozilla/5.0 (INFO5731 Assignment Scraper; +https://github.com)",
    "Accept-Language": "en-US,en;q=0.9"
})

def get_html(url, tries=6, timeout=30):
    """GET with retries for transient errors."""
    for attempt in range(tries):
        r = sess.get(url, timeout=timeout)
        if r.status_code == 200:
            return r.text
        if r.status_code in (429, 500, 502, 503, 504):
            sleep_s = min((2 ** attempt) + random.random(), 30)
            time.sleep(sleep_s)
            continue
        raise RuntimeError(f"HTTP {r.status_code} for {url}\n{r.text[:200]}")
    raise RuntimeError(f"Failed after retries: {url}")

def parse_marketplace_page(html, page_num):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    # Find all links that point to a marketplace action. Using more specific selectors.
    action_links = soup.select("h3 a[href^='/marketplace/actions/'], h2 a[href^='/marketplace/actions/']")

    for a_tag in action_links:
        name = a_tag.get_text(" ", strip=True)
        href = a_tag.get("href", "").strip()
        full_url = urljoin(BASE, href)

        # Navigate up to find the parent container of this action for the description.
        # This assumes a structure like <div class="Box-row">...<h3><a>Name</a></h3><p>Description</p>...</div>
        parent_container = a_tag.find_parent('div', class_='Box-row')
        desc = ""
        if parent_container:
            # Look for a paragraph with a specific class often used for descriptions
            desc_tag = parent_container.select_one('p.f5.color-fg-muted')
            if desc_tag:
                desc = desc_tag.get_text(" ", strip=True)

        # Only add to rows if we successfully extracted a name and a valid URL
        if name and full_url.startswith(BASE + "/marketplace/actions/"):
            rows.append({
                "action_name": name,
                "description": desc,
                "url": full_url,
                "page": page_num
            })
    return rows

all_rows = []
seen_urls = set()

page = 1
while len(all_rows) < TARGET_ITEMS:
    page_url = START_URL + f"&page={page}"
    print("Fetching:", page_url)

    html = get_html(page_url)
    page_rows = parse_marketplace_page(html, page)

    # Stop if no results (end pagination) or if we've seen all items on this page before
    if not page_rows:
        print("No new listings found. Stopping at page:", page)
        break

    added = 0
    for r in page_rows:
        if r["url"] in seen_urls:
            continue
        seen_urls.add(r["url"])
        all_rows.append(r)
        added += 1
        if len(all_rows) >= TARGET_ITEMS:
            break

    print(f"Page {page}: added {added} | total {len(all_rows)}")
    page += 1
    if len(all_rows) < TARGET_ITEMS: # Only sleep if more items are needed
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

raw_df = pd.DataFrame(all_rows).head(TARGET_ITEMS)
print("\nTotal scraped:", len(raw_df))
raw_df.head()

Fetching: https://github.com/marketplace?type=actions&page=1
Page 1: added 20 | total 20
Fetching: https://github.com/marketplace?type=actions&page=2
Page 2: added 20 | total 40
Fetching: https://github.com/marketplace?type=actions&page=3
Page 3: added 20 | total 60
Fetching: https://github.com/marketplace?type=actions&page=4
Page 4: added 20 | total 80
Fetching: https://github.com/marketplace?type=actions&page=5
Page 5: added 20 | total 100
Fetching: https://github.com/marketplace?type=actions&page=6
Page 6: added 20 | total 120
Fetching: https://github.com/marketplace?type=actions&page=7
Page 7: added 20 | total 140
Fetching: https://github.com/marketplace?type=actions&page=8
Page 8: added 20 | total 160
Fetching: https://github.com/marketplace?type=actions&page=9
Page 9: added 20 | total 180
Fetching: https://github.com/marketplace?type=actions&page=10
Page 10: added 20 | total 200
Fetching: https://github.com/marketplace?type=actions&page=11
Page 11: added 20 | total 220
Fetching: 

,action_name,description,url,page
0,TruffleHog OSS,,https://github.com/marketplace/actions/truffle...,1
1,Metrics embed,,https://github.com/marketplace/actions/metrics...,1
2,yq - portable yaml processor,,https://github.com/marketplace/actions/yq-port...,1
3,Super-Linter,,https://github.com/marketplace/actions/super-l...,1
4,Rebuild Armbian and Kernel,,https://github.com/marketplace/actions/rebuild...,1


In [23]:
RAW_CSV = "Panyala_Shresta_GitHubMarketplace_Actions_RAW.csv"
raw_df.to_csv(RAW_CSV, index=False, encoding="utf-8")
print("Saved:", RAW_CSV)
files.download(RAW_CSV)

Saved: Panyala_Shresta_GitHubMarketplace_Actions_RAW.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
## Q4 Part-2 — Preprocess + Data Quality
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_set = set(stopwords.words("english"))
lemm = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [25]:
def basic_clean(text: str) -> str:
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)              # remove HTML tags
    text = re.sub(r"[^A-Za-z\s]", " ", text)        # remove numbers + punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

def tokenize_remove_stopwords(text: str):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_set]
    return tokens

def lemmatize_tokens(tokens):
    return [lemm.lemmatize(t) for t in tokens]

In [28]:
df = raw_df.copy()

# ---- Data Quality ----
# 1) drop duplicates by URL
df = df.drop_duplicates(subset=["url"]).copy()

# 2) drop missing critical fields
df["action_name"] = df["action_name"].astype(str).str.strip()
df["description"] = df["description"].fillna("").astype(str).str.strip()
df["url"] = df["url"].astype(str).str.strip()

df = df[(df["action_name"] != "") & (df["url"].str.startswith("https://github.com/marketplace/"))].copy()

# 3) if description missing, keep row but mark it (optional)
df["desc_missing"] = (df["description"] == "")

# ---- Preprocessing ----
df["desc_clean"] = df["description"].apply(basic_clean)
df["tokens"] = df["desc_clean"].apply(tokenize_remove_stopwords)
df["tokens_lemma"] = df["tokens"].apply(lemmatize_tokens)
df["desc_lemma"] = df["tokens_lemma"].apply(lambda toks: " ".join(toks))

print("After quality checks:", len(df))
df[["action_name","description","desc_clean","desc_lemma","url","page","desc_missing"]].head(3)

After quality checks: 1000


,action_name,description,desc_clean,desc_lemma,url,page,desc_missing
0,TruffleHog OSS,,,,https://github.com/marketplace/actions/truffle...,1,True
1,Metrics embed,,,,https://github.com/marketplace/actions/metrics...,1,True
2,yq - portable yaml processor,,,,https://github.com/marketplace/actions/yq-port...,1,True


In [29]:
CLEAN_CSV = "Panyala_Shresta_GitHubMarketplace_Actions_CLEAN.csv"
df.to_csv(CLEAN_CSV, index=False, encoding="utf-8")
print("Saved:", CLEAN_CSV)
files.download(CLEAN_CSV)

Saved: Panyala_Shresta_GitHubMarketplace_Actions_CLEAN.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [35]:
#PART 1: Collect Tweets Using Tweepy API
!pip -q install tweepy
import tweepy
import pandas as pd
from google.colab import files

In [36]:
BEARER_TOKEN = "PASTE_YOUR_BEARER_TOKEN_HERE"
client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

In [ ]:
# CELL 3: Collect tweets using Sampled Stream (v2)
# This collects random public tweets (no search / no hashtag guarantee)

MAX_TWEETS = 200  # increase later if you want
tweets_data = []

# stream client
class MyStream(tweepy.StreamingClient):
    def on_tweet(self, tweet):
        # Save tweet id + text (username not included in stream by default)
        tweets_data.append({
            "tweet_id": tweet.id,
            "text": tweet.text
        })
        print("Collected:", len(tweets_data))

        if len(tweets_data) >= MAX_TWEETS:
            self.disconnect()

    def on_errors(self, errors):
        print("Stream error:", errors)
        self.disconnect()

stream = MyStream(BEARER_TOKEN)

# Start stream (Ctrl+C to stop manually)
stream.sample()

ERROR:tweepy.streaming:Stream encountered HTTP error: 401
ERROR:tweepy.streaming:HTTP error response text: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status": 401,
  "detail": "Unauthorized"
}
ERROR:tweepy.streaming:Stream encountered HTTP error: 401
ERROR:tweepy.streaming:HTTP error response text: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status": 401,
  "detail": "Unauthorized"
}
ERROR:tweepy.streaming:Stream encountered HTTP error: 401
ERROR:tweepy.streaming:HTTP error response text: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status": 401,
  "detail": "Unauthorized"
}
ERROR:tweepy.streaming:Stream encountered HTTP error: 401
ERROR:tweepy.streaming:HTTP error response text: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status": 401,
  "detail": "Unauthorized"
}
ERROR:tweepy.streaming:Stream encountered HTTP error: 401
ERROR:tweepy.streaming:HTTP error response text: {
  "title": "Unauthorized",
  "type": "about:blank",
  "status":

In [ ]:
# CELL 4: Convert to dataframe + save RAW CSV
df_raw = pd.DataFrame(tweets_data)
print("Total collected:", len(df_raw))
df_raw.head()

RAW_FILE = "Panyala_Shresta_Tweets_Sampled_RAW.csv"
df_raw.to_csv(RAW_FILE, index=False, encoding="utf-8")
print("Saved:", RAW_FILE)
files.download(RAW_FILE)

In [ ]:
##PART 2: Data Cleaning + Data Quality Check

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

The given assignment was highly practical, and it allowed me to realize the principles of the real-world data collection process based on APIs and web scraping. I really liked the preprocessing and NLP sections since they demonstrated how raw data can be purified and changed into a useful information. It provided me with excellent exposure to dealing with actual data rather than theoretical ones.

Nevertheless, I encountered a number of technical difficulties in the process. In Question 1, the 10,000 abstracts of Semantic Scholar were hard to obtain with API rate limits, pagination and token continuation problems. I made several mistakes including 429 (Too Many Requests), and I was forced to make several changes to the code to deal with delays and retries. These restrictions required extensive time to trouble shoot.

The most difficult part was question 5. I kept on getting 401 Unauthorized errors when attempting to access the Twitter API with Tweepy. Authentication and access-level restrictions also made the API unable to give a response to the request so even after going through the tutorial steps and organizing the code correctly, the API could not give out the tweets. Since these were API authorization problems, I could not complete Question 5 to its full potential. The problem was not on the coding logic itself, but probably primarily on the API permissions and restrictions that I cannot control.

Being only a basic/intermediate Python learner, the authentication of APIs, rate limits, and troubleshooting of external system errors proved to be a challenge. Most of these issues were associated with the limitations in the external platform instead of simple syntax errors thereby complicating and taking much time in troubleshooting.

Speaking of the time given, it appeared to be adequate at first, but the API problems and technical limitations came as a sudden surprise, and it was not that easy to accomplish everything and go on the way. More time would have assisted in solving these external problems.

On the whole, this assignment provided me with practical experience in the field of APIs, restrictions, errors, and real-life data processing issues despite the challenges.